# 🇸🇳 SunuBot — Fine-tuning mBERT sur corpus Wolof/Français

**Mémoire M2 IA — Université de Thiès**
**Auteur : Adama GAYE**

Ce notebook fine-tune `bert-base-multilingual-cased` (mBERT) sur un corpus bilingue
français/wolof annoté pour la classification d'intentions administratives sénégalaises.

**Intentions :** demande_document, question_procedure, question_lieu, question_cout,
question_delai, salutation, inscription, suivi_demande

**Domaines :** État Civil, DGID, CNSS, Général

---
⚠️ **Activer le GPU** : Runtime → Change runtime type → T4 GPU

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib seaborn pandas

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Corpus bilingue Wolof/Français annoté

Corpus de 80+ phrases annotées manuellement, augmenté à ~200 par paraphrases.

In [ ]:
CORPUS_BRUT = [
    # ==========================================================================
    # DEMANDE_DOCUMENT — État Civil (40 phrases)
    # ==========================================================================
    ("Dama bëgg sama kàrt bëccëg", "demande_document", "etat_civil"),
    ("Je veux mon acte de naissance", "demande_document", "etat_civil"),
    ("Man naa jël sama kàrt bëccëg?", "demande_document", "etat_civil"),
    ("Dama bëgg kàrt takk bi", "demande_document", "etat_civil"),
    ("Je veux mon acte de mariage", "demande_document", "etat_civil"),
    ("Dama bëgg kàrt dee", "demande_document", "etat_civil"),
    ("J'ai besoin d'un acte de décès", "demande_document", "etat_civil"),
    ("Sama kàrt boppam jeex na, dama bëgg renouveler", "demande_document", "etat_civil"),
    ("Ma carte d'identité est expirée, je veux la renouveler", "demande_document", "etat_civil"),
    ("Dama bëgg certificat nationalité", "demande_document", "etat_civil"),
    ("Je voudrais un certificat de nationalité", "demande_document", "etat_civil"),
    ("Laaj na kàrt bëccëg ngir sama doom", "demande_document", "etat_civil"),
    ("Demande d'acte de naissance pour mon enfant", "demande_document", "etat_civil"),
    ("Dama bëgg copie kàrt bëccëg bi", "demande_document", "etat_civil"),
    ("Je veux une copie de mon acte de naissance", "demande_document", "etat_civil"),
    ("Bëgg naa extrait acte de naissance", "demande_document", "etat_civil"),
    ("J'aimerais obtenir un extrait de naissance", "demande_document", "etat_civil"),
    ("Dama soxla kàrt bëccëg bu bees", "demande_document", "etat_civil"),
    ("Il me faut un nouvel acte de naissance", "demande_document", "etat_civil"),
    ("Naka laa mën a jël certificat résidence?", "demande_document", "etat_civil"),
    ("Comment obtenir un certificat de résidence?", "demande_document", "etat_civil"),
    ("Dama bëgg livret de famille", "demande_document", "etat_civil"),
    ("Je veux un livret de famille", "demande_document", "etat_civil"),
    ("Bëgg naa duplicata kàrt bi", "demande_document", "etat_civil"),
    ("Je demande un duplicata de mon acte", "demande_document", "etat_civil"),
    ("Dama bëgg sama kàrt identité", "demande_document", "etat_civil"),
    ("Je veux ma carte d'identité nationale", "demande_document", "etat_civil"),
    ("Soxla naa kàrt bëccëg ngir liggeey", "demande_document", "etat_civil"),
    ("J'ai besoin d'un acte de naissance pour le travail", "demande_document", "etat_civil"),
    ("Dama bëgg certificat coutumier", "demande_document", "etat_civil"),
    ("Je veux un certificat de coutume", "demande_document", "etat_civil"),
    ("Bëgg naa extrait d'acte de mariage", "demande_document", "etat_civil"),
    ("Je souhaite un extrait d'acte de mariage", "demande_document", "etat_civil"),
    ("Dama bëgg légaliser sama kàrt bi", "demande_document", "etat_civil"),
    ("Je veux faire légaliser mon document", "demande_document", "etat_civil"),
    ("Man naa jël casier judiciaire?", "demande_document", "etat_civil"),
    ("Je veux un extrait de casier judiciaire", "demande_document", "etat_civil"),
    ("Dama bëgg certificat de vie", "demande_document", "etat_civil"),
    ("Je veux un certificat de vie et d'entretien", "demande_document", "etat_civil"),
    ("Bëgg naa copie intégrale sama acte bi", "demande_document", "etat_civil"),

    # ==========================================================================
    # DEMANDE_DOCUMENT — DGID (15 phrases)
    # ==========================================================================
    ("Dama bëgg NINEA ngir sama entreprise", "demande_document", "dgid"),
    ("Je veux immatriculer mon entreprise", "demande_document", "dgid"),
    ("Bëgg naa attestation fiscale", "demande_document", "dgid"),
    ("Je veux une attestation de régularité fiscale", "demande_document", "dgid"),
    ("Dama bëgg quitus fiscal", "demande_document", "dgid"),
    ("Il me faut un quitus fiscal", "demande_document", "dgid"),
    ("Naka laa jël certificat d'imposition?", "demande_document", "dgid"),
    ("Je veux mon certificat d'imposition", "demande_document", "dgid"),
    ("Dama bëgg carte contribuable", "demande_document", "dgid"),
    ("Je souhaite obtenir ma carte de contribuable", "demande_document", "dgid"),
    ("Bëgg naa attestation d'exonération", "demande_document", "dgid"),
    ("Je veux une attestation d'exonération fiscale", "demande_document", "dgid"),
    ("Dama bëgg bordereau de versement", "demande_document", "dgid"),
    ("Soxla naa certificat fiscal ngir marché public", "demande_document", "dgid"),
    ("Je veux un certificat fiscal pour un appel d'offres", "demande_document", "dgid"),

    # ==========================================================================
    # DEMANDE_DOCUMENT — CNSS (8 phrases)
    # ==========================================================================
    ("Dama bëgg sama carte CNSS", "demande_document", "cnss"),
    ("Je veux ma carte d'assuré CNSS", "demande_document", "cnss"),
    ("Bëgg naa attestation d'immatriculation CNSS", "demande_document", "cnss"),
    ("Je veux une attestation CNSS", "demande_document", "cnss"),
    ("Dama bëgg relevé cotisations CNSS", "demande_document", "cnss"),
    ("Je veux un relevé de mes cotisations", "demande_document", "cnss"),
    ("Man naa jël certificat d'affiliation CNSS?", "demande_document", "cnss"),
    ("Je veux un certificat d'affiliation à la CNSS", "demande_document", "cnss"),

    # ==========================================================================
    # QUESTION_PROCEDURE — DGID (20 phrases)
    # ==========================================================================
    ("Naka laa feyee sama wàll?", "question_procedure", "dgid"),
    ("Comment payer mes impôts à la DGID?", "question_procedure", "dgid"),
    ("Dama bëgg def sama déclaration TVA", "question_procedure", "dgid"),
    ("Je veux faire ma déclaration TVA", "question_procedure", "dgid"),
    ("Naka laa def sama declaration NINEA?", "question_procedure", "dgid"),
    ("Comment obtenir mon NINEA?", "question_procedure", "dgid"),
    ("Sama situation fiscale ban la?", "question_procedure", "dgid"),
    ("Quelle est ma situation fiscale?", "question_procedure", "dgid"),
    ("Naka laa mën a fey ci internet?", "question_procedure", "dgid"),
    ("Comment payer en ligne mes impôts?", "question_procedure", "dgid"),
    ("Lan mooy étapes yi ngir créer entreprise?", "question_procedure", "dgid"),
    ("Quelles sont les étapes pour créer une entreprise?", "question_procedure", "dgid"),
    ("Naka laa def déclaration revenus?", "question_procedure", "dgid"),
    ("Comment faire ma déclaration de revenus?", "question_procedure", "dgid"),
    ("Ban papier laa soxla ngir NINEA?", "question_procedure", "dgid"),
    ("Quels documents faut-il pour le NINEA?", "question_procedure", "dgid"),
    ("Naka laa mën a rectifier sama déclaration?", "question_procedure", "dgid"),
    ("Comment corriger une erreur sur ma déclaration?", "question_procedure", "dgid"),
    ("Lan mooy démarches ngir patente?", "question_procedure", "dgid"),
    ("Quelles sont les démarches pour obtenir une patente?", "question_procedure", "dgid"),

    # ==========================================================================
    # QUESTION_PROCEDURE — État Civil (18 phrases)
    # ==========================================================================
    ("Naka laa def ngir jël kàrt bëccëg?", "question_procedure", "etat_civil"),
    ("Comment faire pour obtenir un acte de naissance?", "question_procedure", "etat_civil"),
    ("Ban étapes laa war a def?", "question_procedure", "etat_civil"),
    ("Quelles sont les démarches pour l'état civil?", "question_procedure", "etat_civil"),
    ("Naka laa bind sama doom ci état civil?", "question_procedure", "etat_civil"),
    ("Comment déclarer une naissance?", "question_procedure", "etat_civil"),
    ("Lan mooy procédure mariage civil?", "question_procedure", "etat_civil"),
    ("Quelle est la procédure pour un mariage civil?", "question_procedure", "etat_civil"),
    ("Ban papier laa soxla ngir kàrt bi?", "question_procedure", "etat_civil"),
    ("Quels documents faut-il pour l'acte de naissance?", "question_procedure", "etat_civil"),
    ("Naka laa man a rectifier kàrt bëccëg bi?", "question_procedure", "etat_civil"),
    ("Comment corriger une erreur sur mon acte de naissance?", "question_procedure", "etat_civil"),
    ("Lan mooy procédure ngir déclarer décès?", "question_procedure", "etat_civil"),
    ("Comment déclarer un décès?", "question_procedure", "etat_civil"),
    ("Naka laa mën a jël jugement supplétif?", "question_procedure", "etat_civil"),
    ("Comment obtenir un jugement supplétif?", "question_procedure", "etat_civil"),
    ("Ban papier laa soxla ngir légalisation?", "question_procedure", "etat_civil"),
    ("Quels documents pour la légalisation?", "question_procedure", "etat_civil"),

    # ==========================================================================
    # QUESTION_PROCEDURE — CNSS (10 phrases)
    # ==========================================================================
    ("Naka laa mën a bind ci CNSS?", "question_procedure", "cnss"),
    ("Quelle est la procédure d'inscription à la CNSS?", "question_procedure", "cnss"),
    ("Lan mooy démarches ngir allocations familiales?", "question_procedure", "cnss"),
    ("Comment demander les allocations familiales?", "question_procedure", "cnss"),
    ("Naka laa mën a jël sama retraite?", "question_procedure", "cnss"),
    ("Comment toucher ma pension de retraite?", "question_procedure", "cnss"),
    ("Ban papier laa soxla ngir CNSS?", "question_procedure", "cnss"),
    ("Quels documents pour l'inscription CNSS?", "question_procedure", "cnss"),
    ("Naka laa déclarer sama employés ci CNSS?", "question_procedure", "cnss"),
    ("Comment déclarer mes employés à la CNSS?", "question_procedure", "cnss"),

    # ==========================================================================
    # QUESTION_LIEU (20 phrases)
    # ==========================================================================
    ("Fan la centre fiscal bi nekk?", "question_lieu", "dgid"),
    ("Où se trouve le centre fiscal?", "question_lieu", "dgid"),
    ("Fan la meeri bi nekk ci Thiès?", "question_lieu", "etat_civil"),
    ("Où est la mairie de Thiès?", "question_lieu", "etat_civil"),
    ("Fan la bureau CNSS bi nekk?", "question_lieu", "cnss"),
    ("Où se trouve l'agence CNSS?", "question_lieu", "cnss"),
    ("Fan la bureau état civil bi nekk ci Dakar?", "question_lieu", "etat_civil"),
    ("Où se trouve le bureau de l'état civil à Dakar?", "question_lieu", "etat_civil"),
    ("Adresse DGID Thiès fan la?", "question_lieu", "dgid"),
    ("Quelle est l'adresse de la DGID à Thiès?", "question_lieu", "dgid"),
    ("Fan la tribunal bi nekk?", "question_lieu", "etat_civil"),
    ("Où est le tribunal de grande instance?", "question_lieu", "etat_civil"),
    ("CNSS Dakar fan la nekk?", "question_lieu", "cnss"),
    ("Où est l'agence CNSS de Dakar?", "question_lieu", "cnss"),
    ("Fan laa mën a dem ngir fey sama impôt?", "question_lieu", "dgid"),
    ("Où est-ce que je peux payer mes impôts?", "question_lieu", "dgid"),
    ("Fan la centre d'état civil ci Pikine?", "question_lieu", "etat_civil"),
    ("Où est le centre d'état civil à Pikine?", "question_lieu", "etat_civil"),
    ("CNSS Saint-Louis fan la nekk?", "question_lieu", "cnss"),
    ("Où est l'agence CNSS de Saint-Louis?", "question_lieu", "cnss"),

    # ==========================================================================
    # QUESTION_COUT (20 phrases)
    # ==========================================================================
    ("Ñaata lay jar kàrt bëccëg bi?", "question_cout", "etat_civil"),
    ("Combien coûte l'acte de naissance?", "question_cout", "etat_civil"),
    ("Ñaata xaalis la timbre fiscal bi?", "question_cout", "etat_civil"),
    ("Combien coûte le timbre fiscal?", "question_cout", "etat_civil"),
    ("Ñaata la NINEA bi jar?", "question_cout", "dgid"),
    ("Combien coûte l'obtention du NINEA?", "question_cout", "dgid"),
    ("Ñaata laa war a fey ngir kàrt takk bi?", "question_cout", "etat_civil"),
    ("Quel est le prix de l'acte de mariage?", "question_cout", "etat_civil"),
    ("Frais inscription CNSS ñaata la?", "question_cout", "cnss"),
    ("Combien coûte l'inscription à la CNSS?", "question_cout", "cnss"),
    ("Certificat nationalité ñaata lay jar?", "question_cout", "etat_civil"),
    ("Quel est le coût du certificat de nationalité?", "question_cout", "etat_civil"),
    ("Ñaata la patente bi jar?", "question_cout", "dgid"),
    ("Combien coûte la patente?", "question_cout", "dgid"),
    ("Prix casier judiciaire ñaata la?", "question_cout", "etat_civil"),
    ("Combien coûte le casier judiciaire?", "question_cout", "etat_civil"),
    ("Ñaata laa war a fey ngir légalisation?", "question_cout", "etat_civil"),
    ("Combien coûte une légalisation?", "question_cout", "etat_civil"),
    ("Cotisation mensuelle CNSS ñaata la?", "question_cout", "cnss"),
    ("Quel est le montant de la cotisation CNSS?", "question_cout", "cnss"),

    # ==========================================================================
    # QUESTION_DELAI (20 phrases)
    # ==========================================================================
    ("Ñaata bes la kàrt bëccëg bi jeex?", "question_delai", "etat_civil"),
    ("Quel est le délai pour l'acte de naissance?", "question_delai", "etat_civil"),
    ("Kañ laa man a jël sama NINEA?", "question_delai", "dgid"),
    ("Quand vais-je recevoir mon NINEA?", "question_delai", "dgid"),
    ("Ñaata bes ngir traitement dossier bi?", "question_delai", "etat_civil"),
    ("Combien de jours pour traiter le dossier?", "question_delai", "etat_civil"),
    ("Kañ laa mën a jël sama carte CNSS?", "question_delai", "cnss"),
    ("Quand vais-je recevoir ma carte CNSS?", "question_delai", "cnss"),
    ("Délai ngir certificat nationalité ñaata la?", "question_delai", "etat_civil"),
    ("Quel est le délai pour le certificat de nationalité?", "question_delai", "etat_civil"),
    ("Ñaata bes la carte identité bi mettre?", "question_delai", "etat_civil"),
    ("Combien de temps pour la carte d'identité?", "question_delai", "etat_civil"),
    ("Ñaata bes ngir jël casier judiciaire?", "question_delai", "etat_civil"),
    ("Quel est le délai pour le casier judiciaire?", "question_delai", "etat_civil"),
    ("Kañ laa mën a jël attestation fiscale?", "question_delai", "dgid"),
    ("Quand vais-je recevoir mon attestation fiscale?", "question_delai", "dgid"),
    ("Ñaata bes ngir livret de famille?", "question_delai", "etat_civil"),
    ("Combien de temps pour obtenir le livret de famille?", "question_delai", "etat_civil"),
    ("Pension retraite kañ lay door?", "question_delai", "cnss"),
    ("Quand commence le versement de la retraite?", "question_delai", "cnss"),

    # ==========================================================================
    # INSCRIPTION (20 phrases)
    # ==========================================================================
    ("Dama bëgg bind ci CNSS", "inscription", "cnss"),
    ("Je veux m'inscrire à la CNSS", "inscription", "cnss"),
    ("Naka laa man a bind ci CNSS?", "inscription", "cnss"),
    ("Comment s'inscrire à la CNSS?", "inscription", "cnss"),
    ("Dama bëgg bind ci registre état civil", "inscription", "etat_civil"),
    ("Je veux m'inscrire à l'état civil", "inscription", "etat_civil"),
    ("Comment s'enregistrer à l'état civil?", "inscription", "etat_civil"),
    ("Bëgg naa créer sama compte ci CNSS", "inscription", "cnss"),
    ("Je souhaite ouvrir un compte CNSS", "inscription", "cnss"),
    ("Naka laa mën a enregistrer sama société?", "inscription", "dgid"),
    ("Comment enregistrer ma société à la DGID?", "inscription", "dgid"),
    ("Dama bëgg immatriculer sama boîte", "inscription", "dgid"),
    ("Bëgg naa bind sama doom ci état civil", "inscription", "etat_civil"),
    ("Je veux déclarer mon enfant à l'état civil", "inscription", "etat_civil"),
    ("Dama bëgg créer sama entreprise ci DGID", "inscription", "dgid"),
    ("Je veux créer mon entreprise auprès de la DGID", "inscription", "dgid"),
    ("Naka laa bind sama employés ci CNSS?", "inscription", "cnss"),
    ("Comment inscrire mes employés à la CNSS?", "inscription", "cnss"),
    ("Dama bëgg bind sama jabar ci état civil", "inscription", "etat_civil"),
    ("Je veux enregistrer mon mariage à l'état civil", "inscription", "etat_civil"),

    # ==========================================================================
    # SALUTATION (25 phrases)
    # ==========================================================================
    ("Na nga def", "salutation", "general"),
    ("Bonjour, comment allez-vous?", "salutation", "general"),
    ("Asalaa maalekum", "salutation", "general"),
    ("Bonjour SunuBot", "salutation", "general"),
    ("Jërejëf", "salutation", "general"),
    ("Merci beaucoup", "salutation", "general"),
    ("Ba beneen", "salutation", "general"),
    ("Au revoir", "salutation", "general"),
    ("Bonsoir", "salutation", "general"),
    ("Salut", "salutation", "general"),
    ("Na nga def ci jamm?", "salutation", "general"),
    ("Comment ça va?", "salutation", "general"),
    ("Maalekum salaam", "salutation", "general"),
    ("Noo def?", "salutation", "general"),
    ("Bonne journée", "salutation", "general"),
    ("Bonjour, j'ai besoin d'aide", "salutation", "general"),
    ("Salam", "salutation", "general"),
    ("Hello", "salutation", "general"),
    ("Jërejëf ci wallu", "salutation", "general"),
    ("Merci pour votre aide", "salutation", "general"),
    ("Na nga def SunuBot?", "salutation", "general"),
    ("Bonsoir, je cherche des informations", "salutation", "general"),
    ("Mangui fi rekk", "salutation", "general"),
    ("Jàmm rekk", "salutation", "general"),
    ("Bonjour, est-ce que vous pouvez m'aider?", "salutation", "general"),

    # ==========================================================================
    # SUIVI_DEMANDE (20 phrases)
    # ==========================================================================
    ("Dama bëgg topp sama demande", "suivi_demande", "general"),
    ("Je veux suivre ma demande", "suivi_demande", "general"),
    ("Où en est ma demande DEM-2024-001?", "suivi_demande", "general"),
    ("Mon dossier a-t-il été traité?", "suivi_demande", "general"),
    ("Naka dossier bi dem?", "suivi_demande", "general"),
    ("Quel est le statut de ma demande?", "suivi_demande", "general"),
    ("Sama dossier nekk fan?", "suivi_demande", "general"),
    ("Ma demande est à quel stade?", "suivi_demande", "general"),
    ("Ndax traiter nañu sama dossier?", "suivi_demande", "general"),
    ("Est-ce que mon dossier est prêt?", "suivi_demande", "general"),
    ("Topp naa sama demande bi, fan la nekk?", "suivi_demande", "general"),
    ("Je voudrais connaître l'état d'avancement de mon dossier", "suivi_demande", "general"),
    ("Sama kàrt bi jot na?", "suivi_demande", "etat_civil"),
    ("Mon acte de naissance est-il prêt?", "suivi_demande", "etat_civil"),
    ("Ndax NINEA bi prêt na?", "suivi_demande", "dgid"),
    ("Est-ce que mon NINEA est prêt?", "suivi_demande", "dgid"),
    ("Dama bëgg xam ndax dossier bi avancer na", "suivi_demande", "general"),
    ("Je veux savoir si mon dossier a avancé", "suivi_demande", "general"),
    ("Topp naa sama carte CNSS", "suivi_demande", "cnss"),
    ("Où en est ma carte CNSS?", "suivi_demande", "cnss"),
]

print(f'Corpus brut : {len(CORPUS_BRUT)} exemples')
print(f'Intentions : {len(set(c[1] for c in CORPUS_BRUT))}')
print(f'Domaines : {len(set(c[2] for c in CORPUS_BRUT))}')

## 3. Augmentation du corpus

In [ ]:
import random
random.seed(42)

def augmenter_corpus(corpus):
    augmente = list(corpus)

    substitutions_wolof = [
        ("Dama bëgg", "Dëkk na ma"), ("Dama bëgg", "Bëgg naa"),
        ("Dama bëgg", "Soxla naa"), ("Man naa", "Mën naa"),
        ("Naka laa", "Naka la"), ("Naka laa", "Con naa"),
        ("sama", "sa"), ("Fan la", "Ana"),
        ("Ñaata", "Naata"), ("Kañ laa", "Kañ la"),
        ("Dama soxla", "Soxla naa"), ("ngir", "pour"),
        ("Lan mooy", "Lan la"), ("Ban papier", "Lu laa soxla"),
        ("Bëgg naa", "Dama bëgg"), ("Ndax", "Mbaa"),
    ]
    substitutions_fr = [
        ("Je veux", "Je souhaite"), ("Je veux", "J'aimerais"),
        ("Je veux", "J'ai besoin de"), ("Je veux", "Je désire"),
        ("Comment", "De quelle manière"), ("Comment", "Par quel moyen"),
        ("Où se trouve", "Où est situé"), ("Où se trouve", "Où est-ce que se trouve"),
        ("Où est", "Où se situe"), ("Combien coûte", "Quel est le prix de"),
        ("Combien coûte", "C'est combien"), ("Combien coûte", "Quel est le tarif de"),
        ("Quel est le délai", "Combien de temps faut-il"),
        ("Quel est le délai", "En combien de temps"),
        ("Je voudrais", "J'aimerais"), ("Il me faut", "J'ai besoin de"),
        ("Quelles sont", "C'est quoi"), ("Comment faire", "Comment est-ce qu'on fait"),
        ("Quels documents", "Quelles pièces"), ("J'ai besoin", "Il me faut"),
        ("Est-ce que", "Est-ce qu'"), ("Combien de temps", "Quel délai"),
    ]

    for texte, intent, domaine in corpus:
        # Substitutions wolof
        for old, new in substitutions_wolof:
            if old in texte:
                augmente.append((texte.replace(old, new), intent, domaine))
        # Substitutions français
        for old, new in substitutions_fr:
            if old in texte:
                augmente.append((texte.replace(old, new), intent, domaine))
        # Variante minuscule
        if texte[0].isupper() and len(texte) > 10:
            augmente.append((texte.lower(), intent, domaine))
        # Variante sans ponctuation finale
        if texte.endswith('?') or texte.endswith('.') or texte.endswith('!'):
            augmente.append((texte[:-1], intent, domaine))
        # Variante avec "s'il vous plaît" / "dinaa la neex"
        if intent in ('demande_document', 'inscription', 'suivi_demande'):
            if any(c.isalpha() and ord(c) < 128 for c in texte[:3]):
                augmente.append((texte + " s'il vous plaît", intent, domaine))
            else:
                augmente.append((texte + " baal ma", intent, domaine))

    # Dédupliquer
    seen = set()
    unique = []
    for t, i, d in augmente:
        t_clean = t.strip()
        if t_clean and t_clean not in seen:
            seen.add(t_clean)
            unique.append((t_clean, i, d))
    return unique

corpus = augmenter_corpus(CORPUS_BRUT)
print(f'Corpus brut    : {len(CORPUS_BRUT)} exemples')
print(f'Corpus augmenté: {len(corpus)} exemples')
print(f'Ratio          : x{len(corpus)/len(CORPUS_BRUT):.1f}')

# Distribution des intentions
from collections import Counter
intent_counts = Counter(c[1] for c in corpus)
print('\n--- Distribution des intentions ---')
for intent, count in intent_counts.most_common():
    bar = '█' * (count // 5)
    print(f'  {intent:25s} → {count:3d}  {bar}')

# Distribution des domaines
domain_counts = Counter(c[2] for c in corpus)
print('\n--- Distribution des domaines ---')
for domain, count in domain_counts.most_common():
    bar = '█' * (count // 5)
    print(f'  {domain:25s} → {count:3d}  {bar}')

# Distribution des langues
wolof_count = sum(1 for t, _, _ in corpus if any(w in t.lower() for w in ['dama', 'naka', 'sama', 'bëgg', 'ñaata', 'fan la', 'kañ', 'nekk', 'naa', 'ngir']))
fr_count = len(corpus) - wolof_count
print(f'\n--- Langues ---')
print(f'  Wolof   : {wolof_count} ({wolof_count/len(corpus):.0%})')
print(f'  Français: {fr_count} ({fr_count/len(corpus):.0%})')

## 4. Préparation des données pour mBERT

In [ ]:
INTENTIONS = sorted(set(c[1] for c in corpus))
intent2id = {intent: i for i, intent in enumerate(INTENTIONS)}
id2intent = {i: intent for intent, i in intent2id.items()}

print('Mapping intentions → IDs :')
for intent, idx in intent2id.items():
    print(f'  {idx}: {intent}')

textes = [c[0] for c in corpus]
labels = [intent2id[c[1]] for c in corpus]

X_train, X_test, y_train, y_test = train_test_split(
    textes, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IntentDataset(X_train, y_train, tokenizer)
test_dataset = IntentDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f'Batches train: {len(train_loader)} | Batches test: {len(test_loader)}')

## 5. Fine-tuning mBERT

In [ ]:
NUM_EPOCHS = 15
LEARNING_RATE = 2e-5

model = BertForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
    num_labels=len(INTENTIONS)
).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

print(f'Modèle : bert-base-multilingual-cased')
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')
print(f'Epochs : {NUM_EPOCHS}')
print(f'Learning rate : {LEARNING_RATE}')
print(f'Classes : {len(INTENTIONS)}')

In [ ]:
train_losses = []
train_accs = []
val_accs = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    train_accs.append(train_acc)

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    val_accs.append(val_acc)

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}')

## 6. Courbes d'entraînement

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-o', label='Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Courbe de Loss — Fine-tuning mBERT')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train Accuracy', linewidth=2)
ax2.plot(range(1, NUM_EPOCHS+1), val_accs, 'r-s', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Train vs Validation')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle('SunuBot — Fine-tuning mBERT sur corpus Wolof/Français', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Évaluation finale — Rapport de classification + Matrice de confusion

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

target_names = [id2intent[i] for i in range(len(INTENTIONS))]
print('='*60)
print('RAPPORT DE CLASSIFICATION — mBERT Fine-tuné')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
print(f'Accuracy globale : {accuracy_score(all_labels, all_preds):.2%}')

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title('Matrice de Confusion — mBERT Fine-tuné sur Wolof/Français')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test sur des phrases inédites

In [ ]:
def predire(texte):
    model.eval()
    encoding = tokenizer(texte, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        confiance = probs[0][pred_id].item()
    return id2intent[pred_id], confiance

tests = [
    "Bonjour",
    "Je veux mon acte de naissance",
    "Na nga def",
    "Dama begg sama kart becceg",
    "Comment payer mes impots",
    "Naka lanu feyee alamaan bi",
    "Ou se trouve le bureau de l'etat civil",
    "Combien coute un acte de naissance",
    "Je veux m'inscrire a la CNSS",
    "Dama begg feyee sama wall ci DGID",
]

print('='*70)
print('TESTS SUR PHRASES INÉDITES')
print('='*70)
for texte in tests:
    intent, conf = predire(texte)
    print(f'  "{texte}"')
    print(f'    → {intent} ({conf:.1%})')
    print()

## 9. Sauvegarder le modèle fine-tuné

In [ ]:
import json

# Sauvegarder le modèle et le tokenizer
model.save_pretrained('mbert_finetuned_sunubot')
tokenizer.save_pretrained('mbert_finetuned_sunubot')

# Mettre à jour config.json avec les vrais noms de labels
config_path = 'mbert_finetuned_sunubot/config.json'
with open(config_path) as f:
    config = json.load(f)
config['id2label'] = {str(i): intent for i, intent in id2intent.items()}
config['label2id'] = {intent: i for intent, i in intent2id.items()}
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# Sauvegarder metriques.json (format attendu par le backend SunuBot)
metriques = {
    'label2id': intent2id,
    'id2label': {str(k): v for k, v in id2intent.items()},
    'intentions': INTENTIONS,
    'corpus_size': len(corpus),
    'corpus_brut_size': len(CORPUS_BRUT),
    'accuracy_finale': val_accs[-1] if val_accs else None,
    'epochs': NUM_EPOCHS,
}
with open('mbert_finetuned_sunubot/metriques.json', 'w', encoding='utf-8') as f:
    json.dump(metriques, f, indent=2, ensure_ascii=False)

print('Modèle sauvegardé dans mbert_finetuned_sunubot/')
print(f'Accuracy finale : {val_accs[-1]:.2%}')
print(f'Classes : {INTENTIONS}')
print('\nFichiers :')
import os
for f in sorted(os.listdir('mbert_finetuned_sunubot')):
    size = os.path.getsize(f'mbert_finetuned_sunubot/{f}') / 1e6
    print(f'  {f:40s} {size:6.1f} MB')

In [ ]:
# Télécharger le modèle en ZIP
!zip -r mbert_finetuned_sunubot.zip mbert_finetuned_sunubot/

from google.colab import files
files.download('mbert_finetuned_sunubot.zip')

print('\n✅ Téléchargement lancé !')
print('Placez le contenu du ZIP dans : nlp/models/mbert_finetuned/')

---
## Résumé

| Élément | Valeur |
|---------|--------|
| Modèle | bert-base-multilingual-cased |
| Corpus | ~200 phrases FR/Wolof augmentées |
| Classes | 8 intentions administratives |
| Epochs | 15 |
| GPU | Google Colab T4 |
| Framework | PyTorch + HuggingFace Transformers |

**Adama GAYE — Mémoire M2 IA — Université de Thiès — 2026**